# Data Extraction Flow — Mining and Oil (ANM CFEM, AMB, SCM, SIGBM, DIPEM + ANP wells)

| branch | module | notes |
|---|---|---|
| ANM CFEM | `pipelines.cfem` | single pass, `lookback_meses=6` |
| ANM AMB | `pipelines.amb` | yearly, single pass |
| ANP production by well | `pipelines.anp` | *while loop* until `has_more=False`, 12 months per iteration |
| ANM Mining Registry (SCM) | `pipelines.scm` | snapshot; `range` runs as `incremental` |
| ANM SIGBM dams | `pipelines.sigbm` | snapshot |
| ANM DIPEM investment | `pipelines.dipem` | snapshot |

In [ ]:
from pprint import pprint
import pandas as pd
from pipelines import common as C
from pipelines.flow import run_flow

pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 60)

## Inputs

In [ ]:
mode = "incremental"      # incremental | range | backfill
reference_date = ""         # YYYY-MM-DD; empty = today (America/Sao_Paulo)
start_date = ""             # YYYY-MM or YYYY-MM-DD (required for range; empty backfill = 2015-01)
end_date = ""
force = False               # reload even if ETag/Last-Modified did not change
target_db = "estudo"        # folder in the lake bucket: lake/estudo/ (use another name for tests)
cfg_db = ""                 # where dim_substanciaMineral / dim_alvoExtracao live (Parquet); empty = code defaults

# study extras
dry_run = True              # True = plan + light source check only (HEAD + header)
parallel = False            # True = branches in threads (only 4 cores here, so sequential by default)

## Execute

In [ ]:
spark = None if dry_run else C.get_spark("flow_mineracao_petroleo", cores_max=4)
resumo = run_flow(spark, mode=mode, reference_date=reference_date, start_date=start_date, end_date=end_date,
                  force=force, target_db=target_db, cfg_db=cfg_db, dry_run=dry_run, parallel=parallel)
pd.DataFrame(resumo)

## Source check (dry run only)

Detailed result of each branch (logs above). For a single source, use its own notebook.